# QuantResearch-Playbook 快速上手

本 Notebook 演示如何快速使用本框架进行因子计算和分析。

In [ ]:
# 安装依赖
# !pip install polars numpy scipy

import numpy as np
import polars as pl
np.random.seed(42)

## 1. 生成模拟数据

In [ ]:
n = 1000
data = pl.DataFrame({
    "close": 10 + np.random.randn(n).cumsum() * 0.1,
    "volume": np.abs(np.random.randn(n) * 1e6 + 5e6),
    "amount": np.abs(np.random.randn(n) * 1e8 + 5e8),
    "high": 10 + np.random.randn(n).cumsum() * 0.1 + 0.2,
    "low": 10 + np.random.randn(n).cumsum() * 0.1 - 0.2,
})
print(f"数据形状: {data.shape}")
data.head()

## 2. 计算 CPV 因子

In [ ]:
from qrp.reports.dongwu.cpv_factor import run_cpv_analysis

result = run_cpv_analysis(data)
print(f"IC Mean: {result['ic_metrics'].ic_mean:.4f}")
print(f"ICIR: {result['ic_metrics'].icir:.2f}")
print(f"RankIC Mean: {result['ic_metrics'].rank_ic_mean:.4f}")
print(f"Long/Short Return: {result['long_short_return']:.4%}")
print(f"\n分层收益:")
for q, r in result['quantile_returns'].items():
    print(f"  第{q}分位: {r:.4%}")

## 3. 批量因子对比

In [ ]:
from qrp.reports.dongwu.cpv_factor import run_cpv_advanced
from qrp.core.factor import create_base_factors
from qrp.core.analysis import FactorAnalyzer

# CPV 系列对比
print("=== CPV 系列因子 ===")
for name, m in run_cpv_advanced(data).items():
    print(f"  {name:15s}  IC={m['ic_mean']:.4f}  ICIR={m['icir']:.2f}")

# 基础因子
print("\n=== 基础因子 ===")
pipe = create_base_factors()
for name in pipe.factor_names:
    fac = pipe[name]
    values = fac.calculate(data)
    ic = FactorAnalyzer(data, values).compute_ic()
    print(f"  {name:20s}  IC={ic.ic_mean:.4f}  ICIR={ic.icir:.2f}")

## 4. 回测验证

In [ ]:
from qrp.core.backtest import Backtester

# 生成模拟信号
prices = data["close"]
signals = pl.Series("signal", np.random.choice([-1, 0, 1], len(prices)))

# 运行回测
bt = Backtester()
result = bt.run(prices, signals)

for k, v in result.summary().items():
    print(f"  {k}: {v}")

## 5. 一键自测

In [ ]:
# 在终端中运行:
# python -m self_test.run_all
print("一键自测命令: python -m self_test.run_all")